<a href="https://colab.research.google.com/github/VitorrSan/recomendador-livros/blob/main/02_embeddings_e_similaridade.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q sentence-transformers scikit-learn

In [2]:
import pandas as pd
import numpy as np
import os

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
caminho = "/content/drive/MyDrive/Recomendador de Livros/datasets/livros_limpos.csv"

df_modelo = pd.read_csv(
    caminho,
    dtype={"isbn": "string", "asin": "string"}
)

df_modelo = df_modelo.reset_index(drop=True)

print("Livros carregados:", len(df_modelo))
display(df_modelo.head())

Livros carregados: 1795


,isbn,asin,title,book_cover,author,avg_rating,num_ratings,num_reviews,year,genre,pages,publisher,language,description,url,scraped_at
0,<NA>,B07QPHT8CB,The House in the Cerulean Sea,https://m.media-amazon.com/images/S/compressed...,T.J. Klune,4.36,1020173.0,152296.0,2020.0,Fantasy,393.0,Tor Books,English,A magical island. A dangerous task. A burning ...,https://www.goodreads.com/book/show/45047384-t...,2026-04-06T11:43:18Z
1,9781635574043.0,1635574048,House of Earth and Blood,https://m.media-amazon.com/images/S/compressed...,Sarah J. Maas,4.45,1183081.0,114958.0,2020.0,Fantasy,803.0,Bloomsbury Publishing,English,Bryce Quinlan had the perfect life-working har...,https://www.goodreads.com/book/show/44778083-h...,2026-04-06T11:43:25Z
2,9780765387561.0,0765387565,The Invisible Life of Addie LaRue,https://m.media-amazon.com/images/S/compressed...,V.E. Schwab,4.16,1551956.0,196463.0,2020.0,Fantasy,448.0,Tor Books,English,A Life No One Will Remember. A Story You Will ...,https://www.goodreads.com/book/show/50623864-t...,2026-04-06T11:43:30Z
3,9780525559474.0,0525559477,The Midnight Library,https://m.media-amazon.com/images/S/compressed...,Matt Haig,3.98,2528798.0,252564.0,2020.0,Fiction,288.0,Viking,English,Between life and death there is a library.When...,https://www.goodreads.com/book/show/52578297-t...,2026-04-06T11:43:36Z
4,<NA>,<NA>,American Dirt,https://m.media-amazon.com/images/S/compressed...,Jeanine Cummins,4.37,700358.0,53355.0,2020.0,Fiction,459.0,Flatiron Books,English,"Lydia lives in Acapulco. She has a son, Luca, ...",https://www.goodreads.com/book/show/45046527-a...,2026-04-06T11:43:44Z


In [4]:
df_modelo["genre"] = df_modelo["genre"].fillna("")

df_modelo["texto_combinado"] = (
    "Título: " + df_modelo["title"].astype(str) + ". " +
    "Autor: " + df_modelo["author"].astype(str) + ". " +
    "Gênero: " + df_modelo["genre"].astype(str) + ". " +
    "Descrição: " + df_modelo["description"].astype(str)
)

# Criar identificador para cada livro
df_modelo["book_id"] = np.arange(len(df_modelo))

print(df_modelo["texto_combinado"].iloc[0])

Título: The House in the Cerulean Sea. Autor: T.J. Klune. Gênero: Fantasy. Descrição: A magical island. A dangerous task. A burning secret.Linus Baker leads a quiet, solitary life. At forty, he lives in a tiny house with a devious cat and his old records. As a Case Worker at the Department in Charge Of Magical Youth, he spends his days overseeing the well-being of children in government-sanctioned orphanages.When Linus is unexpectedly summoned by Extremely Upper Management he's given a curious and highly classified assignment: travel to Marsyas Island Orphanage, where six dangerous children reside: a gnome, a sprite, a wyvern, an unidentifiable green blob, a were-Pomeranian, and the Antichrist. Linus must set aside his fears and determine whether or not they’re likely to brin


In [5]:

nome_modelo = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

modelo = SentenceTransformer(nome_modelo)

print("Modelo carregado!")
print("Dimensão dos embeddings:", modelo.get_sentence_embedding_dimension())

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Modelo carregado!
Dimensão dos embeddings: 384


/tmp/ipykernel_718/1434815817.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Dimensão dos embeddings:", modelo.get_sentence_embedding_dimension())


In [6]:
textos = df_modelo["texto_combinado"].tolist()

embeddings = modelo.encode(
    textos,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embeddings gerados!")
print("Formato da matriz:", embeddings.shape)

Batches:   0%|          | 0/57 [00:00<?, ?it/s]

Embeddings gerados!
Formato da matriz: (1795, 384)


In [7]:

# Escolher o primeiro livro do dataset
indice_livro = 0

# Calcular similaridade com todos os livros
similaridades = cosine_similarity(
    embeddings[indice_livro].reshape(1, -1),
    embeddings
)[0]

print("Livro escolhido:")
print(df_modelo.loc[indice_livro, "title"])

Livro escolhido:
The House in the Cerulean Sea


In [20]:

# Ordenar os índices pela maior similaridade
indices_ordenados = np.argsort(similaridades)[::-1]

# Remover o próprio livro da recomendação
indices_ordenados = [
    i for i in indices_ordenados
    if i != indice_livro
]

# Selecionar os cinco primeiros
top_indices = indices_ordenados[:5]

recomendacoes = df_modelo.iloc[top_indices][
    ["book_id", "title", "author", "genre", "avg_rating"]
].copy()

recomendacoes["similaridade"] = similaridades[top_indices]

display(recomendacoes)

,book_id,title,author,genre,avg_rating,similaridade
1325,1325,Somewhere Beyond the Sea,T.J. Klune,Fantasy,4.11,0.834961
13,13,Piranesi,Susanna Clarke,Fantasy,4.21,0.743234
972,972,The Will of the Many,James Islington,Fantasy,4.58,0.735211
149,149,Fable,Adrienne Young,Fantasy,4.00,0.715432
511,511,"The Sneaking, Hiding, Vibrating Creature",Nathan W. Pyle,Picture Books,4.17,0.713283


In [21]:
pasta_resultados = "/content/drive/MyDrive/Recomendador de Livros/resultados"

os.makedirs(pasta_resultados, exist_ok=True)

# Salvar os embeddings
np.save(
    f"{pasta_resultados}/embeddings_livros.npy",
    embeddings
)

# Salvar os metadados na mesma ordem dos embeddings
colunas = [
    "book_id",
    "title",
    "author",
    "genre",
    "avg_rating",
    "year",
    "description",
    "url"
]

df_modelo[colunas].to_csv(
    f"{pasta_resultados}/livros_metadata.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Arquivos exportados com sucesso!")

Arquivos exportados com sucesso!


In [25]:

def recomendar_livros(titulo, quantidade=5):

    # Localizar o livro escolhido
    resultado = df_modelo[
        df_modelo["title"].str.lower() == titulo.lower()
    ]

    if resultado.empty:
        print("Livro não encontrado no dataset.")
        return None

    indice = resultado.index[0]

    # Calcular similaridade com todos os livros
    pontuacoes = cosine_similarity(
        embeddings[indice].reshape(1, -1),
        embeddings
    )[0]

    # Ordenar do mais semelhante ao menos semelhante
    indices = np.argsort(pontuacoes)[::-1]

    # Remover o próprio livro
    indices = [i for i in indices if i != indice]

    # Selecionar recomendações
    indices = indices[:quantidade]

    recomendacoes = df_modelo.iloc[indices][
        ["book_id", "title", "author", "genre", "avg_rating"]
    ].copy()

    recomendacoes["similaridade"] = pontuacoes[indices]

    return recomendacoes.reset_index(drop=True)

In [26]:

titulos_teste = [
    "The House in the Cerulean Sea",
    "The Midnight Library",
    "House of Earth and Blood"
]

df_modelo[
    df_modelo["title"].isin(titulos_teste)
][["book_id", "title", "author"]]

,book_id,title,author
0,0,The House in the Cerulean Sea,T.J. Klune
1,1,House of Earth and Blood,Sarah J. Maas
3,3,The Midnight Library,Matt Haig


In [27]:
livros_carrinho = [0, 1, 3]  # Substitua pelos IDs encontrados

resultado = recomendar_por_carrinho(
    livros_carrinho,
    quantidade=10
)

display(resultado)

,book_id,title,author,genre,avg_rating,similaridade
0,1204,House of Flame and Shadow,Sarah J. Maas,Fantasy,4.24,0.798460
1,1325,Somewhere Beyond the Sea,T.J. Klune,Fantasy,4.11,0.766911
2,1641,Cinder House,Freya Marske,Fantasy,3.77,0.759969
3,719,Our Missing Hearts,Celeste Ng,Fiction,3.79,0.757000
4,674,The Very Secret Society of Irregular Witches,Sangu Mandanna,Fantasy,4.03,0.754063
5,1213,A Fate Inked in Blood,Danielle L. Jensen,Fantasy,3.90,0.753043
6,968,Starling House,Alix E. Harrow,Fantasy,3.78,0.747590
7,1745,The Everlasting,Alix E. Harrow,Fantasy,4.23,0.746673
8,1336,The Life Impossible,Matt Haig,Fiction,3.47,0.746600
9,938,A Soul of Ash and Blood,Jennifer L. Armentrout,Fantasy,3.88,0.743734
